# Visual Caption Generation (LLaVA-Med / Qwen2.5-VL)

**Stage 1** of the visual captioning pipeline: generate an initial biomedical
caption directly from each figure MinerU2.5 extracted from a PDF.

- **LLaVA-Med** generates captions for clinical photos and microscopy
  (`type == "image"`). It was trained on PMC figure-caption pairs, so its
  output is visually grounded for this content.
- **Qwen2.5-VL** handles diagrams, charts, and tables (`type in {"chart",
  "table", "diagram"}`), since LLaVA-Med's 7B backbone is weaker on
  structured layouts and labeled components.
- MinerU2.5 already tags content type in `*_content_list.json`, so the
  router is just a conditional on that field — no separate classifier
  needed.

**Known limitation of Stage 1 alone:** LLaVA-Med can hallucinate medical
terms, mis-identify anatomical structures, or use imprecise clinical
language, since its 7B LLM backbone isn't specifically trained on clinical
guidelines. A verification/refinement stage (Stage 2) is out of scope for
this notebook.

**Hardware note:** this runs on Apple Silicon (MPS, no CUDA). LLaVA-Med
is not natively `transformers`-compatible and hard-pins
`transformers==4.36.2`, which conflicts with the `transformers` version
this project needs for Qwen2.5-VL. It's vendored at `vendor/LLaVA-Med`
with its own isolated venv and invoked as a subprocess — see
`vendor/README.md`. Both models are loaded one at a time to fit in 24GB
unified memory.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from captioning.router import VisualCaptionRouter

PROJECT_ROOT

/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student')

## Load MinerU2.5 output for one PDF

In [2]:
PDF_STEM = "Anatomy of Face and Oral Cavity - Basic of DEMN.pdf"
OUTPUT_DIR = PROJECT_ROOT / "output" / PDF_STEM / "hybrid_auto"
CONTENT_LIST_PATH = OUTPUT_DIR / f"{PDF_STEM}_content_list.json"

with open(CONTENT_LIST_PATH) as f:
    content_list = json.load(f)

visual_items = [
    item for item in content_list
    if item.get("type") in {"image", "chart", "table", "diagram"} and item.get("img_path")
]
len(visual_items), visual_items[0]

(102,
 {'type': 'image',
  'img_path': 'images/b6b3baa9cb90ab1ea5bf18406845ade4c753c010a64d101e570d1f0920cc3689.jpg',
  'image_caption': [],
  'image_footnote': [],
  'content': 'Anatomical illustration of human head and neck muscles, showing detailed anatomical structures without any text or labels.',
  'sub_type': 'natural_image',
  'bbox': [0, 0, 500, 998],
  'page_idx': 0})

## Router smoke test (one image per branch)

In [3]:
router = VisualCaptionRouter()

by_type = {}
for item in visual_items:
    by_type.setdefault(item["type"], item)

by_type.keys()

dict_keys(['image', 'table'])

In [4]:
# Test the LLaVA-Med path (clinical photo / microscopy) on one "image"-type item
sample = by_type.get("image")
if sample:
    image_path = OUTPUT_DIR / sample["img_path"]
    result = router.caption(str(image_path), sample["type"])
    print(json.dumps(result, indent=2))
else:
    print("no 'image'-type item found in this PDF")

KeyboardInterrupt: 

In [ ]:
# Test the Qwen2.5-VL path (structured visual) on one "chart" or "table"-type item
sample = by_type.get("chart") or by_type.get("table") or by_type.get("diagram")
if sample:
    image_path = OUTPUT_DIR / sample["img_path"]
    result = router.caption(str(image_path), sample["type"])
    print(json.dumps(result, indent=2))
else:
    print("no structured-type item found in this PDF")

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

## Batch run over all visuals in the PDF

Runs LLaVA-Med items first, then Qwen2.5-VL items, to minimize model
load/unload churn (rather than alternating per-item).

In [ ]:
llava_items = [i for i in visual_items if router.route(i["type"]) == "llava_med"]
qwen_items = [i for i in visual_items if router.route(i["type"]) == "qwen_vl"]

results = []

for item in llava_items:
    image_path = OUTPUT_DIR / item["img_path"]
    results.append(router.caption(str(image_path), item["type"]))

for item in qwen_items:
    image_path = OUTPUT_DIR / item["img_path"]
    results.append(router.caption(str(image_path), item["type"]))

len(results)

In [ ]:
results_path = OUTPUT_DIR / f"{PDF_STEM}_stage1_captions.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

results_path